# Smart Grid Estimation — Test Notebook

This notebook demonstrates the automatic block size detection for natural images.

**Why not FFT?**  
FFT-based grid detection (like perfectPixel) looks for periodic peaks in images that are *already pixelated*. Our inputs are natural photos with no existing grid pattern, so FFT finds nothing useful.

**Our approach:**  
1. **Edge density analysis** — measures how detailed the image is  
2. **Multi-scale SSIM** — pixelates at many block sizes and measures structural quality loss  
3. **Elbow detection** — finds the largest block size before quality drops significantly

In [ ]:
import sys, os
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Add parent path so we can import smart_grid
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from grid_estimation.smart_grid import (
    estimate_block_size,
    estimate_block_size_fast,
    _pixelate,
    _compute_edge_density,
    _evaluate_block_sizes
)

## 1. Load test images

In [ ]:
test_dir = os.path.join('..', 'test_cases')
image_files = [f for f in os.listdir(test_dir) if f.endswith(('.jpg', '.png'))]
print(f"Found {len(image_files)} test images: {image_files}")

## 2. Run estimation on all test images

In [ ]:
results = {}
for fname in image_files:
    path = os.path.join(test_dir, fname)
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        print(f"Skipping {fname} (failed to load)")
        continue
    
    bs_ssim = estimate_block_size(img)
    bs_fast = estimate_block_size_fast(img)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    ed = _compute_edge_density(gray)
    
    results[fname] = {
        'img': img,
        'size': f"{img.shape[1]}x{img.shape[0]}",
        'edge_density': ed,
        'block_ssim': bs_ssim,
        'block_fast': bs_fast
    }
    print(f"{fname:30s}  size={results[fname]['size']:>10s}  "
          f"edge_density={ed:.4f}  block(SSIM)={bs_ssim:3d}  block(fast)={bs_fast:3d}")

## 3. Visual comparison: Original vs Pixelated

In [ ]:
for fname, info in results.items():
    img = info['img']
    bs = info['block_ssim']
    
    pix = _pixelate(img, bs)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[0].set_title(f"Original ({info['size']})")
    axes[0].axis('off')
    
    axes[1].imshow(cv2.cvtColor(pix, cv2.COLOR_BGR2RGB))
    axes[1].set_title(f"Pixelated (block={bs}, edge_density={info['edge_density']:.4f})")
    axes[1].axis('off')
    
    fig.suptitle(fname, fontsize=14)
    plt.tight_layout()
    plt.show()

## 4. SSIM curve with debug mode
Pick one image and run with `debug=True` to see the full SSIM curve and elbow point.

In [ ]:
# Pick the first available image for detailed debug view
sample_name = list(results.keys())[0]
sample_img = results[sample_name]['img']
print(f"Running debug on: {sample_name}")
bs = estimate_block_size(sample_img, debug=True)
print(f"Estimated block size: {bs}")

## 5. Integration example with pixelator
Shows how to use `estimate_block_size` in the pixelation pipeline.

In [ ]:
def pixelate_auto(img):
    """
    Full auto pixelation: detect block size, then pixelate.
    """
    block_size = estimate_block_size(img)
    h, w = img.shape[:2]
    new_w = w // block_size
    new_h = h // block_size
    
    # Pixelate to small size (this is the final output)
    small = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
    return small, block_size


# Demo
sample_img = results[list(results.keys())[0]]['img']
small_img, bs = pixelate_auto(sample_img)
print(f"Block size: {bs}, Output: {small_img.shape[1]}x{small_img.shape[0]}")

plt.figure(figsize=(8, 8))
plt.imshow(cv2.cvtColor(small_img, cv2.COLOR_BGR2RGB), interpolation='nearest')
plt.title(f"Auto-pixelated output ({small_img.shape[1]}x{small_img.shape[0]}, block={bs})")
plt.axis('off')
plt.show()